# LiteRT-LM NPU Compiler

This notebook demonstrates how to compile a `.litertlm` model package for NPU execution using the `litert_torch` library. 

Ahead-of-Time (AOT) compilation optimizes the execution of generative models (like Gemma) on NPUs (Qualcomm Snapdragon, MediaTek Dimensity) by compiling target subgraphs into native NPU payloads.

## 1. Installation

First, install the `litert-torch-nightly` package from PyPI.

In [ ]:
# Install LiteRT Torch Nightly from PyPI
!pip install litert-torch-nightly

## 2. NPU Compiler SDK Setup

To compile models, the environment needs access to the vendor-specific NPU compiler library (e.g. QNN for Qualcomm, Neuron for MediaTek). 

You can install the compiler helper libraries (if available) or download the vendor SDKs manually.

In [ ]:
# Option A: Install dynamic SDK wrapper packages (if available)
# !pip install ai-edge-litert-sdk-qualcomm
# !pip install ai-edge-litert-sdk-mediatek

# Option B: Manual SDK setup
# If you have manually downloaded the Qualcomm QNN SDK or MediaTek Neuron SDK,
# uncomment and update the path below to expose the SDK libraries to the compiler:
# import os
# os.environ['LD_LIBRARY_PATH'] = '/path/to/qnn_or_neuron_sdk/lib/x86_64-linux-clang:' + os.environ.get('LD_LIBRARY_PATH', '')

## 3. Run Compilation (CLI)

You can run the NPU compiler directly from the command line using the Python module execution. 

Make sure to specify:
*   `--backend`: `qualcomm` or `mediatek`
*   `--soc_model`: The target SoC model ID (case-insensitive, e.g. `SM8850`, `MT6993`)

In [ ]:
!python -m litert_torch.generative.export_hf.experimental.litert_lm_npu_compiler.litert_lm_npu_compiler_main \
  --input_litertlm="gemma.litertlm" \
  --output_litertlm="gemma_compiled.litertlm" \
  --backend="qualcomm" \
  --soc_model="SM8850"


## 4. Run Compilation (Python API)

Alternatively, you can import and call the compiler function programmatically in your model packaging scripts:

In [ ]:
from litert_torch.generative.export_hf.experimental.litert_lm_npu_compiler.litert_lm_npu_compiler import compile_litertlm

compile_litertlm(
    input_litertlm="gemma.litertlm",
    output_litertlm="gemma_compiled.litertlm",
    backend="qualcomm",
    soc_model="SM8850",
    # disable_weight_sharing=False, # (Qualcomm only, defaults to False)
    # disable_aux_compilation=False,  # (Qualcomm only, defaults to False)
)
print("Compilation and repacking completed!")

## 5. Verification (Optional)

You can verify that the subgraphs were compiled to NPU by checking if the output TFLite files contain custom `DISPATCH_OP` operators (which represent QNN or Neuron partitions).

In [ ]:
import pathlib
import sys
import tempfile
import tomllib

from litert_lm_builder import litertlm_peek
from tensorflow.lite.python import schema_py_generated as schema_fb


def verify_litertlm_compilation(litertlm_path):
  with tempfile.TemporaryDirectory() as temp_dir:
    temp_dir_path = pathlib.Path(temp_dir)
    print(f'Unpacking {litertlm_path} for inspection...')
    litertlm_peek.peek_litertlm_file(litertlm_path, temp_dir, sys.stdout)

    toml_path = temp_dir_path / 'model.toml'
    with open(toml_path, 'r') as f:
      toml_data = tomllib.loads(f.read())

    if 'section' not in toml_data:
      return

    for section in toml_data['section']:
      if section.get('section_type') == 'TFLiteModel':
        model_type = section.get('model_type')
        data_path = temp_dir_path / section.get('data_path')

        # Read the TFLite model flatbuffer
        with open(data_path, 'rb') as f:
          model_bytes = f.read()
        model = schema_fb.Model.GetRootAsModel(model_bytes, 0)

        # Count Dispatch OPs in the model
        dispatch_ops_count = 0
        for i in range(model.SubgraphsLength()):
          subgraph = model.Subgraphs(i)
          for j in range(subgraph.OperatorsLength()):
            op = subgraph.Operators(j)
            opcode = model.OperatorCodes(op.OpcodeIndex())
            # DISPATCH_OP is represented as a custom operator
            if opcode.BuiltinCode() == schema_fb.BuiltinOperator.CUSTOM:
              custom_code = opcode.CustomCode()
              if custom_code == b'DISPATCH_OP':
                dispatch_ops_count += 1

        print(f"Model Section: {model_type} ({section.get('data_path')})")
        print(f'  - Total Subgraphs: {model.SubgraphsLength()}')
        print(
            '  - Total NPU Compiled Partitions (DISPATCH_OP):'
            f' {dispatch_ops_count}'
        )
        print('-' * 50)


# Verify the compiled file
# verify_litertlm_compilation("gemma_compiled.litertlm")